In [ ]:
# Generative Models: From Codes to Samples
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part5/19-generative.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part5').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part5')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

import math

import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import torch
from torch import Tensor, nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
torch.set_num_threads(6)
torch.manual_seed(6050)

navy, orange, green, wine = "#232D4B", "#E57200", "#2E7D32", "#722F37"

assert _BOOK_ROOT.is_dir()

**Plan**

1. Define the reusable helpers: `gaussian_kl`, `gaussian_elbo`, and `gaussian_density`.
2. Prepare the inputs and fixed settings for the example.
3. Verify the exact Gaussian ELBO gap.

In [ ]:
# [1]
def gaussian_kl(
    mean_q: float, variance_q: float, mean_p: float, variance_p: float
) -> float:
    return 0.5 * (
        math.log(variance_p / variance_q)
        + (variance_q + (mean_q - mean_p) ** 2) / variance_p
        - 1.0
    )


def gaussian_elbo(
    x: float, likelihood_variance: float, mean_q: float, variance_q: float
) -> float:
    expected_log_likelihood = -0.5 * (
        math.log(2.0 * math.pi * likelihood_variance)
        + ((x - mean_q) ** 2 + variance_q) / likelihood_variance
    )
    return expected_log_likelihood - gaussian_kl(mean_q, variance_q, 0.0, 1.0)


def gaussian_density(grid: Tensor, mean: float, variance: float) -> Tensor:
    return torch.exp(-(grid - mean).square() / (2.0 * variance)) / math.sqrt(
        2.0 * math.pi * variance
    )


# [2]
x_observed = 1.2
likelihood_variance = 0.36
posterior_mean = x_observed / (1.0 + likelihood_variance)
posterior_variance = likelihood_variance / (1.0 + likelihood_variance)
log_evidence = -0.5 * (
    math.log(2.0 * math.pi * (1.0 + likelihood_variance))
    + x_observed**2 / (1.0 + likelihood_variance)
)
exact_elbo = gaussian_elbo(
    x_observed, likelihood_variance, posterior_mean, posterior_variance
)
mismatch_mean, mismatch_variance = 0.35, 0.75
mismatch_elbo = gaussian_elbo(
    x_observed, likelihood_variance, mismatch_mean, mismatch_variance
)
posterior_gap = gaussian_kl(
    mismatch_mean, mismatch_variance, posterior_mean, posterior_variance
)

# [3]
print(f"posterior mean:     {posterior_mean:.12f}")
print(f"posterior variance: {posterior_variance:.12f}")
print(f"log evidence:       {log_evidence:.12f}")
print(f"exact ELBO:         {exact_elbo:.12f}")
print(f"mismatched ELBO:    {mismatch_elbo:.12f}")
print(f"evidence - ELBO:    {log_evidence - mismatch_elbo:.12f}")
print(f"KL(q || posterior): {posterior_gap:.12f}")

**Plan**

1. Define the reusable `js_divergence` helper.
2. Prepare the inputs and fixed settings for the example.
3. Audit finite GAN coverage and generator saturation.

In [ ]:
# [1]
def js_divergence(p: Tensor, q: Tensor) -> Tensor:
    midpoint = 0.5 * (p + q)
    return 0.5 * (
        (p * torch.log(p / midpoint)).sum()
        + (q * torch.log(q / midpoint)).sum()
    )


# [2]
data_mass = torch.tensor([0.45, 0.45, 0.10])
generator_masses = {
    "covered": torch.tensor([0.40, 0.45, 0.15]),
    "collapsed": torch.tensor([0.02, 0.96, 0.02]),
}
gan_rows = {}
for name, generator_mass in generator_masses.items():
    optimal_discriminator = data_mass / (data_mass + generator_mass)
    value = (
        (data_mass * torch.log(optimal_discriminator)).sum()
        + (generator_mass * torch.log(1.0 - optimal_discriminator)).sum()
    )
    divergence = js_divergence(data_mass, generator_mass)
    identity = -math.log(4.0) + 2.0 * divergence
    gan_rows[name] = optimal_discriminator, divergence, value, identity

# [3]
logit_grid = torch.linspace(-6.0, 6.0, 300)
discriminator_output = torch.sigmoid(logit_grid)
minimax_magnitude = discriminator_output
nonsaturating_magnitude = 1.0 - discriminator_output

print("candidate  JSD            V(D*,G)        -log(4)+2JSD    error")
for name, (_, divergence, value, identity) in gan_rows.items():
    print(
        f"{name:9s}  {divergence.item():.12f}  {value.item():.12f}  "
        f"{identity.item():.12f}  {(value - identity).abs().item():.2e}"
    )
for logit in [-6.0, -2.0, 0.0, 2.0]:
    probability = torch.sigmoid(torch.tensor(logit)).item()
    print(
        f"logit {logit:4.1f}: D={probability:.6f}, "
        f"minimax |grad|={probability:.6f}, "
        f"non-saturating |grad|={1.0 - probability:.6f}"
    )

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Audit a forward noise schedule and its direct marginal.

In [ ]:
# [1]
diffusion_steps = 100
diffusion_beta = torch.linspace(1e-4, 0.1, diffusion_steps)
diffusion_alpha = 1.0 - diffusion_beta
diffusion_alpha_bar = torch.cumprod(diffusion_alpha, dim=0)
diffusion_bars_with_zero = torch.cat((torch.ones(1), diffusion_alpha_bar))

# [2]
torch.manual_seed(6050)
audit_x0 = torch.randn(10_000)
step_noises = torch.randn(diffusion_steps, audit_x0.numel())
sequential_state = audit_x0.clone()
accumulated_noise = torch.zeros_like(audit_x0)
for step in range(diffusion_steps):
    sequential_state = (
        torch.sqrt(diffusion_alpha[step]) * sequential_state
        + torch.sqrt(diffusion_beta[step]) * step_noises[step]
    )
    accumulated_noise = (
        torch.sqrt(diffusion_alpha[step]) * accumulated_noise
        + torch.sqrt(diffusion_beta[step]) * step_noises[step]
    )
direct_state = (
    torch.sqrt(diffusion_alpha_bar[-1]) * audit_x0 + accumulated_noise
)
effective_epsilon = accumulated_noise / torch.sqrt(
    1.0 - diffusion_alpha_bar[-1]
)

print("t  alpha_bar      signal          noise")
for time_index in [1, 10, 50, 100]:
    alpha_bar_at_time = diffusion_alpha_bar[time_index - 1]
    print(
        f"{time_index:3d}  {alpha_bar_at_time.item():.12f}  "
        f"{torch.sqrt(alpha_bar_at_time).item():.12f}  "
        f"{torch.sqrt(1.0 - alpha_bar_at_time).item():.12f}"
    )
print(
    "largest sequential/direct difference at T:",
    f"{(sequential_state - direct_state).abs().max().item():.2e}",
)
print(
    "effective epsilon mean / variance:",
    f"{effective_epsilon.mean().item():.6f}",
    f"{effective_epsilon.var(unbiased=False).item():.6f}",
)

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Pin reverse-posterior coefficients and the final no-noise branch.

In [ ]:
# [1]
previous_alpha_bar = torch.cat((torch.ones(1), diffusion_alpha_bar[:-1]))
posterior_variance = (
    diffusion_beta * (1.0 - previous_alpha_bar)
    / (1.0 - diffusion_alpha_bar)
)
x0_coefficient = (
    torch.sqrt(previous_alpha_bar) * diffusion_beta
    / (1.0 - diffusion_alpha_bar)
)
xt_coefficient = (
    torch.sqrt(diffusion_alpha) * (1.0 - previous_alpha_bar)
    / (1.0 - diffusion_alpha_bar)
)

print("t  x0 coefficient  xt coefficient  posterior variance")
for time_index in [1, 2, 10, 50, 100]:
    index = time_index - 1
    print(
        f"{time_index:3d}  {x0_coefficient[index].item():.12f}  "
        f"{xt_coefficient[index].item():.12f}  "
        f"{posterior_variance[index].item():.12f}"
    )
# [2]
assert posterior_variance[0].item() == 0.0
print("t=1 stochastic-noise coefficient:", f"{posterior_variance[0].sqrt().item():.1f}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable helpers: `ScalarNoisePredictor` and `mixture_batch`.
3. Define the reusable helpers: `oracle_noise_mean` and `fit_diffusion_ablation`.
4. Train a tiny diffusion denoiser with and without timestep conditioning.
5. Report or visualize the measured result.

In [ ]:
# [1]
training_beta = diffusion_beta.float()
training_alpha = 1.0 - training_beta
training_alpha_bar = torch.cumprod(training_alpha, dim=0)
training_previous_bar = torch.cat((
    torch.ones_like(training_alpha_bar[:1]), training_alpha_bar[:-1]
))
training_posterior_variance = (
    training_beta * (1.0 - training_previous_bar)
    / (1.0 - training_alpha_bar)
)


# [2]
class ScalarNoisePredictor(nn.Module):
    def __init__(self, time_conditioned: bool) -> None:
        super().__init__()
        self.time_conditioned = time_conditioned
        self.network = nn.Sequential(
            nn.Linear(2, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x_t: Tensor, timestep: Tensor) -> Tensor:
        if self.time_conditioned:
            scaled_time = 2.0 * timestep[:, None].float() / (diffusion_steps - 1) - 1.0
        else:
            scaled_time = torch.zeros_like(x_t)
        model_input = torch.cat((x_t, scaled_time), dim=1)
        return self.network(model_input)


torch.manual_seed(6050)
conditioned_initialization = ScalarNoisePredictor(True)
torch.manual_seed(6050)
no_time_initialization = ScalarNoisePredictor(False)
assert sum(p.numel() for p in conditioned_initialization.parameters()) == 4_417
assert all(
    torch.equal(conditioned, no_time)
    for conditioned, no_time in zip(
        conditioned_initialization.state_dict().values(),
        no_time_initialization.state_dict().values(),
    )
)
del conditioned_initialization, no_time_initialization


def mixture_batch(generator: torch.Generator, batch_size: int) -> Tensor:
    component = 2 * torch.randint(0, 2, (batch_size, 1), generator=generator) - 1
    noise = torch.randn(batch_size, 1, generator=generator, dtype=torch.float32)
    return 2.0 * component.float() + 0.5 * noise


# [3]
def oracle_noise_mean(x_t: Tensor, timestep: Tensor) -> Tensor:
    alpha_bar_at_time = training_alpha_bar[timestep, None]
    root_bar = torch.sqrt(alpha_bar_at_time)
    noisy_variance = 0.25 * alpha_bar_at_time + 1.0 - alpha_bar_at_time
    component_means = torch.cat((-2.0 * root_bar, 2.0 * root_bar), dim=1)
    log_weights = -0.5 * (x_t - component_means).square() / noisy_variance
    weights = torch.softmax(log_weights, dim=1)
    posterior_gain = 0.25 * root_bar / noisy_variance
    clean_component_means = torch.cat((
        -2.0 + posterior_gain * (x_t + 2.0 * root_bar),
        2.0 + posterior_gain * (x_t - 2.0 * root_bar),
    ), dim=1)
    expected_x0 = (weights * clean_component_means).sum(dim=1, keepdim=True)
    return (x_t - root_bar * expected_x0) / torch.sqrt(1.0 - alpha_bar_at_time)


def fit_diffusion_ablation(
    seed: int, time_conditioned: bool
) -> tuple[dict[str, float], Tensor]:
    torch.manual_seed(seed)
    train_generator = torch.Generator().manual_seed(seed + 19_000)
    model = ScalarNoisePredictor(time_conditioned).float()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

    for _ in range(5_000):
        clean = mixture_batch(train_generator, 512)
        timestep = torch.randint(diffusion_steps, (512,), generator=train_generator)
        epsilon = torch.randn(512, 1, generator=train_generator, dtype=torch.float32)
        alpha_bar_at_time = training_alpha_bar[timestep, None]
        noisy = (
            torch.sqrt(alpha_bar_at_time) * clean
            + torch.sqrt(1.0 - alpha_bar_at_time) * epsilon
        )
        loss = (model(noisy, timestep) - epsilon).square().mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    evaluation_generator = torch.Generator().manual_seed(91_900)
    evaluation_clean = mixture_batch(evaluation_generator, 50_000)
    evaluation_timestep = torch.randint(
        diffusion_steps, (50_000,), generator=evaluation_generator
    )
    evaluation_epsilon = torch.randn(
        50_000, 1, generator=evaluation_generator, dtype=torch.float32
    )
    evaluation_bar = training_alpha_bar[evaluation_timestep, None]
    evaluation_noisy = (
        torch.sqrt(evaluation_bar) * evaluation_clean
        + torch.sqrt(1.0 - evaluation_bar) * evaluation_epsilon
    )

    sampling_generator = torch.Generator().manual_seed(seed + 29_000)
    generated = torch.randn(
        20_000, 1, generator=sampling_generator, dtype=torch.float32
    )
    with torch.no_grad():
        evaluation_prediction = model(evaluation_noisy, evaluation_timestep)
        evaluation_mse = (
            evaluation_prediction - evaluation_epsilon
        ).square().mean().item()
        oracle_mse = (
            oracle_noise_mean(evaluation_noisy, evaluation_timestep)
            - evaluation_epsilon
        ).square().mean().item()

        for index in range(diffusion_steps - 1, -1, -1):
            timestep = torch.full((generated.shape[0],), index, dtype=torch.long)
            predicted_epsilon = model(generated, timestep)
            reverse_mean = (
                generated
                - training_beta[index]
                / torch.sqrt(1.0 - training_alpha_bar[index])
                * predicted_epsilon
            ) / torch.sqrt(training_alpha[index])
            if index > 0:
                reverse_noise = torch.randn(
                    generated.shape, generator=sampling_generator,
                    dtype=torch.float32,
                )
            else:
                reverse_noise = torch.zeros_like(generated)
            generated = (
                reverse_mean
                + torch.sqrt(training_posterior_variance[index]) * reverse_noise
            )

    target_generator = torch.Generator().manual_seed(seed + 39_000)
    target = mixture_batch(target_generator, 20_000)[:, 0]
    generated_vector = generated[:, 0]
    wasserstein_one = (
        torch.sort(generated_vector).values - torch.sort(target).values
    ).abs().mean().item()
    metrics = {
        "noise MSE": evaluation_mse,
        "oracle MSE": oracle_mse,
        "mean": generated_vector.mean().item(),
        "standard deviation": generated_vector.std(unbiased=False).item(),
        "positive mass": (generated_vector > 0).float().mean().item(),
        "central mass": (generated_vector.abs() < 1).float().mean().item(),
        "Wasserstein-1": wasserstein_one,
    }
    return metrics, generated_vector


torch.set_num_threads(1)
diffusion_results: dict[str, list[dict[str, float]]] = {
    "time conditioned": [], "no time": []
}
example_samples = {}
# [4]
conditions = [("time conditioned", True), ("no time", False)]
for condition_name, time_conditioned in conditions:
    for diffusion_seed in range(6050, 6055):
        metrics, generated_samples = fit_diffusion_ablation(
            diffusion_seed, time_conditioned
        )
        diffusion_results[condition_name].append(metrics)
        if diffusion_seed == 6050:
            example_samples[condition_name] = generated_samples
torch.set_num_threads(6)

true_standard_deviation = math.sqrt(4.25)
normal_cdf = lambda value: 0.5 * (1.0 + math.erf(value / math.sqrt(2.0)))
true_central_mass = normal_cdf(-2.0) - normal_cdf(-6.0)

# [5]
print("condition          metric              mean         sample SD")
for condition_name, rows in diffusion_results.items():
    for metric_name in [
        "noise MSE", "oracle MSE", "mean", "standard deviation",
        "positive mass", "central mass", "Wasserstein-1",
    ]:
        values = torch.tensor([row[metric_name] for row in rows])
        print(
            f"{condition_name:17s}  {metric_name:18s}  "
            f"{values.mean().item():.9f}  {values.std().item():.9f}"
        )
print(f"target standard deviation: {true_standard_deviation:.9f}")
print(f"target positive mass:      {0.5:.9f}")
print(f"target central mass:       {true_central_mass:.9f}")
print(f"alpha_bar_T:               {training_alpha_bar[-1].item():.12f}")